<center>
    <p style="text-align:center">
        <img alt="phoenix logo" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://arize.com/docs/phoenix/">Docs</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://arize-ai.slack.com/join/shared_invite/zt-2w57bhem8-hq24MB6u7yE_ZF_ilOYSBw#/shared-invite/email">Community</a>
    </p>
</center>
<h1 align="center">Tool Calling Evals</h1>

The purpose of this notebook is:

- to evaluate the multi-step LLM logic involved in tool calling,
- to provide an experimental framework for users to iterate and improve on the default evaluation template.

## Install Dependencies and Import Libraries

[Youtube tutorial](https://www.youtube.com/watch?v=Rsu-UZ1ZVZU&pp=ygUZcGhvZW5peCB0cmFjaW5nIGZ1bmN0aW9ucw%3D%3D)

In [1]:
# %pip install langchain langchain-openai openinference-instrumentation-langchain

In [2]:
# %pip install langchain_google_vertexai google-cloud-aiplatform

In [3]:
# %pip install -qq "arize-phoenix>=8.8.0" "arize-phoenix-otel>=0.8.0" llama-index-llms-openai openai gcsfs nest_asyncio langchain langchain-openai openinference-instrumentation-langchain

In [4]:
# import os
# from getpass import getpass

# if not (openai_api_key := os.getenv("OPENAI_API_KEY")):
#     openai_api_key = getpass("🔑 Enter your OpenAI API key: ")

# os.environ["OPENAI_API_KEY"] = openai_api_key

In [5]:
from google.cloud.aiplatform import init as init_vertexai
PROJECT_ID = "seequent-labs-dev"  
init_vertexai(project=PROJECT_ID, location="us-central1")

In [6]:
from langchain_google_vertexai import ChatVertexAI
model_name = "gemini-2.5-flash"

model = ChatVertexAI(
    model=model_name,
    temperature=0,
)


In [7]:

# from phoenix.evals import (
#     GeminiModel,  # Use GeminiModel instead of VertexAIModel
# )
# model = GeminiModel(
#     model="gemini-2.0-flash",
#     project=PROJECT_ID,
#     location="us-central1"
# )

In [8]:
import nest_asyncio
import pandas as pd

from phoenix.evals import (
    TOOL_CALLING_PROMPT_RAILS_MAP,
    TOOL_CALLING_PROMPT_TEMPLATE,
    # OpenAIModel,
    llm_classify,
)

nest_asyncio.apply()

/Users/stepan.lavrinenko/phoenix/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Generate App Usage Data

Let's begin by generating some data to use for our evaluation. Let's pretend we have a chatbot for an ecommerce company that is armed with a set of functions to lookup products and orders.

In [9]:
GEN_TEMPLATE = """
You are an assistant that generates complex customer service questions. You will try to answer the question with the tool if possible,
do your best to answer, ask for more information only if needed.
The questions should often involve:

Please reference the product names, the product details, product IDS and product information.

Multiple Categories: Questions that could logically fall into more than one category (e.g., combining product details with a discount code).
Vague Details: Questions with limited or vague information that require clarification to categorize correctly.
Mixed Intentions: Queries where the customer’s goal or need is unclear or seems to conflict within the question itself.
Indirect Language: Use of indirect or polite phrasing that obscures the direct need or request (e.g., using "I was wondering if..." or "Perhaps you could help me with...").
For specific categories:

Track Package: Include vague timing references (e.g., "recently" or "a while ago") instead of specific dates.
Product Comparison and Product Search: Include generic descriptors without specific product names or IDs (e.g., "high-end smartphones" or "energy-efficient appliances").
Apply Discount Code: Include questions about discounts that might apply to hypothetical or past situations, or without mentioning if they have made a purchase.
Product Details: Ask for comparisons or details that involve multiple products or categories ambiguously (e.g., "Tell me about your range of electronics that are good for home office setups").
Examples of More Challenging Questions
Multiple Categories

"I recently bought a samsung 106i smart phone, and I was wondering if there's a way to check what deals I might have missed or if my order is on its way?"
"Could you tell me if the samsung 15H adapater in my last order are covered under warranty and if they have shipped yet?"
Vague Details

"There's an issue with one of the Vizio 14Y TV I think I bought last month—what should I do?"
"I need help with a iPhone 16H I ordered, or maybe I'm just looking for something new. Can you help?"
Mixed Intentions

"I'm not sure if I should ask for a refund or just find out when it will arrive. What do you suggest?"
"Could you help me decide whether to upgrade my product or just track the current one?"
Indirect Language

"I was wondering if you might assist me in figuring out a problem I have with an order, or maybe it's more of a query?"
"Perhaps you could help me understand the benefits of your premium products compared to the regular ones?"

Some questions should be straightforward uses of the provided functions

Respond with a list, one question per line. Do not include any numbering at the beginning of each line. Do not include any category headings.
Generate 20 questions.
"""

In [10]:
# model = OpenAIModel(model="gpt-4o", max_tokens=1300)

In [11]:
# resp = model(GEN_TEMPLATE)
resp = model.invoke(GEN_TEMPLATE).content

KeyboardInterrupt: 

In [ ]:
split_response = resp.strip().split("\n")

questions_df = pd.DataFrame(split_response, columns=["questions"])
print(questions_df)

                                            questions
0   I recently bought a Samsung 106i smartphone, a...
1   Could you tell me if the Samsung 15H adapter i...
2   There's an issue with one of the Vizio 14Y TV ...
3   I need help with a iPhone 16H I ordered, or ma...
4   I'm not sure if I should ask for a refund or j...
5   Could you help me decide whether to upgrade my...
6   I was wondering if you might assist me in figu...
7   Perhaps you could help me understand the benef...
8   I'm trying to understand the features of the n...
9   I was hoping to get some details on the ProVie...
10  I'm having trouble with a SmartHome Hub X that...
11  I'm looking for a high-end smartphone, perhaps...
12  My Zenith Z-series laptop (model Z-1500) has b...
13  I'm wondering if I should return the SoundWave...
14  I was hoping you could shed some light on the ...
15  I'm just trying to get a sense of whether the ...
16  Could you please provide me with the full spec...
17  I'm looking for an energ

# Define a tool-calling agent with Langchain

Now we'll define the chatbot agent using Langchain to attach the functions as tools.

In [ ]:
from langchain import hub
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI

import phoenix as px
from phoenix.otel import register

### Connect to Phoenix

We'll also enable tracing with Phoenix using our Langchain auto-instrumentor to capture telemetry that we can later evaluate.

This code will connect you to an online version of Phoenix, at app.phoenix.arize.com. If you're self-hosting Phoenix, be sure to change your Collector Endpoint below, and remove the API Key.

In [ ]:
# if not (phoenix_api_key := os.getenv("PHOENIX_API_KEY")):
#     phoenix_api_key = getpass("🔑 Enter your Phoenix API key: ")

# os.environ["PHOENIX_API_KEY"] = phoenix_api_key
# os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = "https://app.phoenix.arize.com/"
# os.environ["PHOENIX_CLIENT_HEADERS"] = f"api_key={phoenix_api_key}"

# os.environ["PHOENIX_PROJECT_NAME"] = "Tool Calling Eval"

# tracer_provider = register(auto_instrument=True, project_name="Tool Calling Eval")

In [ ]:
import os

os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = "http://localhost:6006"


from phoenix.otel import register

os.environ["PHOENIX_PROJECT_NAME"] = "evaluate_tool_calling"

# configure the Phoenix tracer
tracer_provider = register(
  project_name="evaluate_tool_calling", # Default is 'default'
  auto_instrument=True # Auto-instrument your app based on installed OI dependencies
)

  # Import the automatic instrumentor from OpenInference
from openinference.instrumentation.langchain import LangChainInstrumentor

# Finish automatic instrumentation
LangChainInstrumentor().instrument(tracer_provider=tracer_provider)
from openinference.instrumentation import using_attributes, using_tags


Attempting to instrument while already instrumented


🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: evaluate_tool_calling
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: localhost:4317
|  Transport: gRPC
|  Transport Headers: {'user-agent': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



Now we'll define our basic functions using pydantic. The actual logic of the functions doesn't matter in this evaluation scenario, since we won't be evaluating anything beyond the function generation step.

In [ ]:
## function definitions using pydantic decorator


@tool
def product_comparison(product_a_id: str, product_b_id: str) -> dict:
    """
    Compare features of two products.

    Parameters:
    product_a_id (str): The unique identifier of Product A.
    product_b_id (str): The unique identifier of Product B.

    Returns:
    dict: A dictionary containing the comparison of the two products.
    """

    if product_a_id == "" or product_b_id == "":
        return {"error": "missing product id"}

    # Implement the function logic here
    return {"comparison": "Similar"}


@tool
def product_details(product_id: str) -> dict:
    """
    Get detailed features on one product.

    Parameters:
    product_id (str): The unique identifier of the Product.

    Returns:
    dict: A dictionary containing product details.
    """

    if product_id == "":
        return {"error": "missing product id"}

    # Implement the function logic here
    return {"name": "Product Name", "price": "$12.50", "Availability": "In Stock"}


@tool
def apply_discount_code(order_id: int, discount_code: str) -> dict:
    """
    Applies a discount code to an order.

    Parameters:
    order_id (str): The unique identifier of the order.
    discount_code (str): The discount code to apply.

    Returns:
    dict: A dictionary containing the updated order details.
    """

    if order_id == "" or discount_code == "":
        return {"error": "missing order id or discount code"}

    # Implement the function logic here
    return {"applied": "True"}


@tool
def product_search(
    query: str,
    category: str = None,
    min_price: float = 0.0,
    max_price: float = None,
    page: int = 1,
    page_size: int = 20,
) -> dict:
    """
    Search for products based on criteria.

    Parameters:
    query (str): The search query string.
    category (str, optional): The category to filter the search. Default is None.
    min_price (float, optional): The minimum price of the products to search. Default is 0.
    max_price (float, optional): The maximum price of the products to search. Default is None.
    page (int, optional): The page number for pagination. Default is 1.
    page_size (int, optional): The number of results per page. Default is 20.

    Returns:
    dict: A dictionary containing the search results and pagination info.
    """

    if query == "":
        return {"error": "missing query"}

    # Implement the function logic here
    return {"results": [], "pagination": {"total": 0, "page": 1, "page_size": 20}}


@tool
def customer_support(issue_type: str) -> dict:
    """
    Get contact information for customer support regarding an issue.

    Parameters:
    issue_type (str): The type of issue (e.g., billing, technical support).

    Returns:
    dict: A dictionary containing the contact information for customer support.
    """

    if issue_type == "":
        return {"error": "missing issue type"}

    # Implement the function logic here
    return {"contact": issue_type}


@tool
def track_package(tracking_number: int) -> dict:
    """
    Track the status of a package based on the tracking number.

    Parameters:
    tracking_number (str): The tracking number of the package.

    Returns:
    dict: A dictionary containing the tracking status of the package.
    """
    if tracking_number == "":
        return {"error": "missing tracking number"}

    # Implement the function logic here
    return {"status": "Delivered"}


tools = [
    product_comparison,
    product_search,
    customer_support,
    track_package,
    apply_discount_code,
    product_details,
]

In [ ]:
# llm = ChatOpenAI(model="gpt-4o")
llm=model
# prompt = hub.pull("hwchase17/openai-functions-agent")
# agent = create_tool_calling_agent(llm, tools, prompt)
# agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
llm = llm.bind_tools(tools)


# Run Chain on each question

With our agent defined, we can now run it across each generated question.

In [ ]:
questions_df["response"] = questions_df["questions"].apply(
    lambda x: llm.invoke(x)
)

In [ ]:
questions_df


,questions,response
0,"I recently bought a Samsung 106i smartphone, a...",content='I can help you check the status of yo...
1,Could you tell me if the Samsung 15H adapter i...,"content=""I can help you with that, but I'll ne..."
2,There's an issue with one of the Vizio 14Y TV ...,content='' additional_kwargs={'function_call':...
3,"I need help with a iPhone 16H I ordered, or ma...","content=""I can help with both!\n\nIf you're lo..."
4,I'm not sure if I should ask for a refund or j...,content='Please provide me with the tracking n...
5,Could you help me decide whether to upgrade my...,"content=""I can help with both! To recommend an..."
6,I was wondering if you might assist me in figu...,content='I can help with that. Can you please ...
7,Perhaps you could help me understand the benef...,"content=""I can help you with that! To compare,..."
8,I'm trying to understand the features of the n...,"content='To help you with the promotions, coul..."
9,I was hoping to get some details on the ProVie...,content='' additional_kwargs={'function_call':...


## Check tool calling usage

In [ ]:
for q in questions_df["response"]:
    print(q.tool_calls)

[]
[]
[{'name': 'customer_support', 'args': {'issue_type': 'technical support'}, 'id': '52571854-888e-4e80-8d07-f8cfaef56e10', 'type': 'tool_call'}]
[]
[]
[]
[]
[]
[]
[{'name': 'product_details', 'args': {'product_id': 'PV-4K'}, 'id': '941a66bf-71e9-4e1b-83a0-be7203e1c402', 'type': 'tool_call'}]
[]
[{'name': 'product_search', 'args': {'query': 'high-end smartphone', 'category': 'smartphone'}, 'id': '7338877d-e93d-446a-90f1-3a64a30e339b', 'type': 'tool_call'}]
[]
[]
[]
[{'name': 'product_details', 'args': {'product_id': 'PV-4K'}, 'id': 'f9e7e7c0-2d1f-481f-a432-6da998779db4', 'type': 'tool_call'}]
[]
[{'name': 'product_search', 'args': {'query': 'energy-efficient smart refrigerator', 'category': 'refrigerators'}, 'id': '86df900b-0737-41b5-a02a-d646564fbe73', 'type': 'tool_call'}]
[]
[]


# Evaluate Tool Calls

Now that we have some example runs of our agent to analyze, we can start the evaluation process. We'll start by exporting all of those spans from Phoenix

In [ ]:
from phoenix.trace import SpanEvaluations
from phoenix.trace.dsl import SpanQuery

Since we'll only be evaluating the inputs, outputs, and function call columns, let's extract those into an easier to use df. Helpfully, Phoenix provides a method to query your span data and directly export only the values you care about.

In [ ]:
query = (
    SpanQuery()
    .where(
        # Filter for the `LLM` span kind.
        # The filter condition is a string of valid Python boolean expression.
        "span_kind == 'LLM'",
    )
    .select(
        # Extract and rename the following span attributes
        question="llm.input_messages",
        outputs="llm.output_messages",
        tool_call="llm.tool_calls",
    )
)
trace_df = px.Client().query_spans(query, project_name="evaluate_tool_calling")
trace_df["tool_call"] = trace_df["tool_call"].fillna("No tool used")

/Users/stepan.lavrinenko/phoenix/.venv/lib/python3.12/site-packages/phoenix/utilities/client.py:51: UserWarning: ⚠️⚠️ The Phoenix server (10.13.2) and client (11.5.0) versions are severely mismatched. Upgrade  either the client or server to ensure API compatibility ⚠️⚠️
  warnings.warn(


In [ ]:
trace_df

,question,outputs,tool_call
context.span_id,,,
ed0a7509dc764711,"[{'message': {'role': 'system', 'content': 'Yo...","[{'message': {'role': 'assistant', 'content': ...",No tool used
39fae1dd0097fb8c,"[{'message': {'role': 'system', 'content': 'Yo...","[{'message': {'role': 'assistant', 'content': ...",No tool used
12a6e120f8f8a4c7,"[{'message': {'role': 'system', 'content': 'Yo...",[{'message': {'function_call_arguments_json': ...,No tool used
ee980ac7f5fd21ae,"[{'message': {'role': 'system', 'content': 'Yo...","[{'message': {'role': 'assistant', 'content': ...",No tool used
fe3c92596b9bb33b,"[{'message': {'role': 'system', 'content': 'Yo...","[{'message': {'role': 'assistant', 'content': ...",No tool used
...,...,...,...
d3b554070701c88b,"[{'message': {'role': 'user', 'content': 'I'm ...",[{'message': {'function_call_arguments_json': ...,No tool used
2ddc233e3d6bf36e,"[{'message': {'role': 'user', 'content': 'Coul...","[{'message': {'role': 'assistant', 'content': ...",No tool used
5287b9d33f19461a,"[{'message': {'role': 'user', 'content': 'I'm ...",[{'message': {'function_call_arguments_json': ...,No tool used


In [ ]:
trace_df['tool_call'].value_counts()

tool_call
No tool used    116
Name: count, dtype: int64

In [ ]:
def get_tool_call(outputs):
    if outputs[0].get("message").get("tool_calls"):
        return (
            outputs[0]
            .get("message")
            .get("tool_calls")[0]
            .get("tool_call")
            .get("function")
            .get("name")
        )
    else:
        return "No tool used"


trace_df["tool_call"] = trace_df["outputs"].apply(get_tool_call)

We'll also need to pass in our tool definitions to the evaluator:

In [ ]:
tool_definitions = ""

for current_tool in tools:
    tool_definitions += f"""
    {current_tool.name}: {current_tool.description}
    """

print(tool_definitions)


    product_comparison: Compare features of two products.

Parameters:
product_a_id (str): The unique identifier of Product A.
product_b_id (str): The unique identifier of Product B.

Returns:
dict: A dictionary containing the comparison of the two products.
    
    product_search: Search for products based on criteria.

Parameters:
query (str): The search query string.
category (str, optional): The category to filter the search. Default is None.
min_price (float, optional): The minimum price of the products to search. Default is 0.
max_price (float, optional): The maximum price of the products to search. Default is None.
page (int, optional): The page number for pagination. Default is 1.
page_size (int, optional): The number of results per page. Default is 20.

Returns:
dict: A dictionary containing the search results and pagination info.
    
    customer_support: Get contact information for customer support regarding an issue.

Parameters:
issue_type (str): The type of issue (e.g.

Next, we define the evaluator model to use

In [ ]:
trace_df["tool_definitions"] = tool_definitions

In [ ]:
# # Check if DataFrame has data
# print(f"DataFrame shape: {trace_df.shape}")
# print(f"DataFrame columns: {trace_df.columns.tolist()}")
# print(f"First few rows:\n{trace_df.head()}")

# # Check for empty DataFrame
# if trace_df.empty:
#     print("ERROR: trace_df is empty!")


In [ ]:
# from phoenix.evals import TOOL_CALLING_PROMPT_TEMPLATE

# # Print the template to see what columns it expects
# print("Template content:")
# print(TOOL_CALLING_PROMPT_TEMPLATE)

# # Check if your DataFrame has the required columns
# required_columns = ["input", "output", "tools"]  # Common tool calling columns
# missing_columns = [col for col in required_columns if col not in trace_df.columns]
# if missing_columns:
#     print(f"Missing columns: {missing_columns}")


In [ ]:
from phoenix.evals import (
    GeminiModel,  # Use Phoenix's VertexAI model instead
)

In [ ]:
# eval_model = OpenAIModel(model="gpt-4o")
# Set up the VertexAI model for evaluation
eval_model = GeminiModel(
    model="gemini-2.5-flash",  # or your preferred Gemini model
    project=PROJECT_ID,  # Replace with your GCP project ID
    location="us-central1"  # Replace with your preferred region
)

/Users/stepan.lavrinenko/phoenix/.venv/lib/python3.12/site-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


And we're ready to call our evaluator! The method below takes in the dataframe of traces to evaluate, our built in evaluation prompt, the eval model to use, and a rails object to snap responses from our model to a set of binary classification responses.

We'll also instruct our model to provide explanations for its responses.

In [ ]:




rails = list(TOOL_CALLING_PROMPT_RAILS_MAP.values())

response_classifications = llm_classify(
    dataframe=trace_df,
    template=TOOL_CALLING_PROMPT_TEMPLATE,
    model=eval_model,
    rails=rails,
    provide_explanation=True,
)

response_classifications["score"] = response_classifications.apply(
    lambda x: 1 if x["label"] == "correct" else 0, axis=1
)


/var/folders/z8/r7gby1gs0ts1089jpzfjwfjc0000gp/T/ipykernel_64739/4118721497.py:3: DeprecationWarning: `dataframe` argument is deprecated; use `data` instead
  response_classifications = llm_classify(
llm_classify |█████████▉| 115/116 (99.1%) | ⏳ 01:56<00:00 |  1.29it/s

In [ ]:
response_classifications

,label,explanation,exceptions,execution_status,execution_seconds,score
context.span_id,,,,,,
ed0a7509dc764711,correct,EXPLANATION: The user's question has two parts...,[],COMPLETED,7.096647,1
39fae1dd0097fb8c,correct,EXPLANATION: The user's question asks about th...,[],COMPLETED,6.317493,1
12a6e120f8f8a4c7,incorrect,EXPLANATION: The user is reporting an issue wi...,[],COMPLETED,3.973631,0
ee980ac7f5fd21ae,incorrect,EXPLANATION: The conversation history clearly ...,[],COMPLETED,3.946102,0
fe3c92596b9bb33b,correct,EXPLANATION: The user's request is ambiguous a...,[],COMPLETED,7.739358,1
...,...,...,...,...,...,...
d3b554070701c88b,correct,EXPLANATION: The user is asking for an evaluat...,[],COMPLETED,3.850672,1
2ddc233e3d6bf36e,incorrect,"EXPLANATION: The user's question asks for ""ful...",[],COMPLETED,3.129703,0
5287b9d33f19461a,incorrect,EXPLANATION: The user is asking for product re...,[],COMPLETED,3.022698,0


Finally, we'll export these responses back into Phoenix to view them in the UI.

In [ ]:
px.Client().log_evaluations(
    SpanEvaluations(eval_name="Tool Calling Eval", dataframe=response_classifications),
)

/Users/stepan.lavrinenko/phoenix/.venv/lib/python3.12/site-packages/phoenix/utilities/client.py:51: UserWarning: ⚠️⚠️ The Phoenix server (10.13.2) and client (11.5.0) versions are severely mismatched. Upgrade  either the client or server to ensure API compatibility ⚠️⚠️
  warnings.warn(
llm_classify |██████████| 116/116 (100.0%) | ⏳ 01:57<00:00 |  1.01s/it
/Users/stepan.lavrinenko/phoenix/.venv/lib/python3.12/site-packages/phoenix/utilities/client.py:51: UserWarning: ⚠️⚠️ The Phoenix server (10.13.2) and client (11.5.0) versions are severely mismatched. Upgrade  either the client or server to ensure API compatibility ⚠️⚠️
  warnings.warn(


From here, you could iterate on different agent logic and prompts to improve performance, or you could further decompose the evaluation into individual steps looking first at Routing, then Parameter Extraction, then Code Generation to determine where to focus.

![Tool Calling Evaluation Results](https://storage.googleapis.com/arize-phoenix-assets/assets/images/tool-calling-nb-result.png)

Happy building!